# Extract Boundary Tokens - Google Drive Version

Utilise les fichiers directement depuis votre Google Drive

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Drive
drive.mount('/content/gdrive')
print("✓ Google Drive montée")

# Vérifier les fichiers
drive_path = Path('/content/gdrive/My Drive/Khabar-segmentation')
if drive_path.exists():
    print(f"✓ Dossier du projet trouvé: {drive_path}")
else:
    print(f"✗ Dossier non trouvé. Cherchez le chemin exact dans votre Drive")
    print(f"   Chemin attendu: /content/gdrive/My Drive/Khabar-segmentation")

## Cell 2: Define Paths & Install Dependencies

In [ ]:
from pathlib import Path
import os

# ⚠️ MODIFIER CES CHEMINS SELON VOTRE DRIVE
# Exemple: '/content/gdrive/My Drive/Khabar-segmentation'
PROJECT_DIR = '/content/gdrive/My Drive/Khabar-segmentation'

CORPUS_PATH = Path(PROJECT_DIR) / 'data/processed/kitab_uqala_reference_corpus.txt'
PREDICTIONS_PATH = Path(PROJECT_DIR) / 'results/camelbert_kitab_uqala_raw_inference.json'
OUTPUT_PATH = Path(PROJECT_DIR) / 'results/camelbert_boundary_tokens_clean.json'

print("Chemins configurés:")
print(f"  Corpus: {CORPUS_PATH}")
print(f"  Prédictions: {PREDICTIONS_PATH}")
print(f"  Output: {OUTPUT_PATH}")

print("\nVérification des fichiers:")
print(f"  Corpus existe: {CORPUS_PATH.exists()}")
print(f"  Prédictions existent: {PREDICTIONS_PATH.exists()}")

# Install dependencies
!pip install transformers torch --quiet
print("\n✓ Dépendances installées")

## Cell 3: Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

print("[1/5] Charger le tokenizer...")
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"      ✓ {model_name}")

## Cell 4: Load Corpus & Predictions

In [ ]:
import json

print("[2/5] Charger le corpus...")
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    text = f.read()
print(f"      ✓ {len(text):,} caractères")

print("\n[3/5] Charger les prédictions...")
with open(PREDICTIONS_PATH, 'r', encoding='utf-8') as f:
    predictions = json.load(f)['inference_results']['predictions']
print(f"      ✓ {len(predictions):,} prédictions")

## Cell 5: Tokenize & Extract

In [ ]:
print("[4/5] Tokenizing...")
tokens = tokenizer.tokenize(text)
print(f"      ✓ {len(tokens):,} tokens")

# Handle mismatch
if len(tokens) != len(predictions):
    print(f"      ⚠️  Mismatch: {len(tokens)} vs {len(predictions)}")
    min_len = min(len(tokens), len(predictions))
    tokens = tokens[:min_len]
    predictions = predictions[:min_len]
    print(f"      Using {min_len}")

print("\n[5/5] Extracting boundary tokens...")
boundary_tokens = [t for t, p in zip(tokens, predictions) if p == 1]
boundary_indices = [i for i, p in enumerate(predictions) if p == 1]
print(f"      ✓ {len(boundary_tokens):,} boundary tokens found")

## Cell 6: Save Results

In [ ]:
# Create structured results
results = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus.txt',
        'corpus_size_chars': len(text),
        'corpus_size_tokens': len(tokens),
        'model': 'CAMeL-Lab/bert-base-arabic-camelbert-msa',
        'total_boundary_tokens': len(boundary_tokens),
        'boundary_percentage': round(100 * len(boundary_tokens) / len(tokens), 2),
    },
    'statistics': {
        'total_tokens': len(tokens),
        'boundary_tokens_count': len(boundary_tokens),
        'non_boundary_tokens': len(tokens) - len(boundary_tokens),
        'boundary_ratio': round(len(boundary_tokens) / len(tokens), 4),
    },
    'boundary_tokens': boundary_tokens,
    'boundary_indices': boundary_indices,
}

# Save to Google Drive
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✓ Sauvegardé: {OUTPUT_PATH}")
print(f"\nRésumé:")
print(f"  Boundary tokens: {len(boundary_tokens):,}")
print(f"  Pourcentage: {results['metadata']['boundary_percentage']}%")
print(f"  Corpus: {len(text):,} caractères")

## Cell 7: Preview Results

In [ ]:
print("Premiers 30 boundary tokens:")
for i, token in enumerate(boundary_tokens[:30], 1):
    print(f"  {i:2d}. {token}")

print(f"\n... ({len(boundary_tokens) - 30:,} tokens restants)")